In [1]:
%pip install pyarrow



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import meteostat as ms
from meteostat import Point, daily
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import sqlite3
from datetime import date


In [3]:
# 1) Download the stations database (once)
STATIONS_DB_URL = "https://data.meteostat.net/stations.db"
STATIONS_DB_FILE = "stations.db"

with requests.get(STATIONS_DB_URL, stream=True) as r:
    r.raise_for_status()
    with open(STATIONS_DB_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# 2) Open the DB and get all station IDs in France
conn = sqlite3.connect(STATIONS_DB_FILE)
cur = conn.cursor()

# country code 'FR' for France
cur.execute("SELECT id,region,latitude, longitude FROM stations WHERE country = 'FR'")
rows = cur.fetchall()
conn.close()

# Opret en Pandas DataFrame direkte fra resultatet for nem håndtering
# Kolonnerne vi skal bruge er 'id', 'latitude' og 'longitude'
df_stations = pd.DataFrame(rows, columns=['station_id', 'region','lat', 'lon'])

print(f"DataFrame created with {len(df_stations)} stations and coordinates.")
print(df_stations.head(20))



DataFrame created with 223 stations and coordinates.
   station_id region      lat     lon
0       07502      B  44.5333 -1.1333
1       LFSM0      I  47.4870  6.7905
2       07554      K  44.5333  3.4500
3       07630      N  43.6333  1.3667
4       07311      T  46.2500 -1.5667
5       07499      V  45.9167  6.8667
6       07157      J  49.0167  2.5333
7       07179      M  48.7833  5.4833
8       07117      E  48.8258 -3.4731
9       07374      C  46.1667  3.4000
10      LFFS0      G  49.1500  4.5333
11      07201      E  47.9667 -4.1667
12      07535     LO  44.7500  1.4000
13      07265      D  47.8000  3.5500
14      07622      N  43.6833  0.6000
15      07235      R  47.9333  0.2000
16      07503      B  44.4333 -1.2500
17      07031      P  49.3667  0.1667
18      07203      E  47.6500 -3.5000
19      07028      Q  49.5167  0.0667


### Using KNN

In [4]:

def _normalize_region_code(x) -> str:
    """
    Simpel normalisering af regionskoder:
    - trim/upper
    - fjern '.0'
    (ingen 2A/2B-specialtilfælde som for departementer)
    """
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s.endswith(".0"):
        s = s[:-2]
    return s


MAINLAND_REGION_CODES = {
    "11", "24", "27", "28", "32",
    "44", "52", "53", "75", "76",
    "84", "93", "94",  # 94 = Corse (include it; remove if you don't want Corsica)
}


def build_region_station_weights_idw(
    df_stations: pd.DataFrame,
    gdf_regions: gpd.GeoDataFrame,
    reg_code_col: str = "code",
    station_id_col: str = "station_id",
    lon_col: str = "lon",
    lat_col: str = "lat",
    K: int = 5,
    p: float = 2.0,
    crs_stations: str = "EPSG:4326",
    crs_distance: str = "EPSG:2154",
) -> pd.DataFrame:
    # --- 1) Stations GeoDataFrame ---
    req = {station_id_col, lon_col, lat_col}
    missing = req - set(df_stations.columns)
    if missing:
        raise ValueError(f"df_stations mangler kolonner: {missing}")

    df_st = df_stations[[station_id_col, lon_col, lat_col]].copy()
    df_st = df_st.dropna(subset=[station_id_col, lon_col, lat_col])

    gdf_st = gpd.GeoDataFrame(
        df_st,
        geometry=gpd.points_from_xy(df_st[lon_col], df_st[lat_col]),
        crs=crs_stations
    )

    # --- 2) Regioner: kode + repræsentativt punkt ---
    if reg_code_col not in gdf_regions.columns:
        raise ValueError(
            f"reg_code_col='{reg_code_col}' findes ikke i gdf_regions. "
            f"Tilgængelige kolonner: {list(gdf_regions.columns)}"
        )

    gdf_reg = gdf_regions[[reg_code_col, "geometry"]].copy()
    gdf_reg["region_code"] = gdf_reg[reg_code_col].apply(_normalize_region_code)

    # 💡 Kun fastlandsregioner (inkl. Korsika)
    gdf_reg = gdf_reg[gdf_reg["region_code"].isin(MAINLAND_REGION_CODES)]

    # Projectér til meter
    gdf_reg = gdf_reg.to_crs(crs_distance)
    gdf_st = gdf_st.to_crs(crs_distance)

    # Repræsentativt punkt pr. region
    gdf_reg["rep_point"] = gdf_reg.geometry.representative_point()

    # --- 3) Krydsprodukt (region x station) og afstand ---
    reg_tbl = gdf_reg[["region_code", "rep_point"]].copy()
    st_tbl = gdf_st[[station_id_col, "geometry"]].rename(columns={station_id_col: "station_id"}).copy()

    reg_tbl["key"] = 1
    st_tbl["key"] = 1
    pairs = reg_tbl.merge(st_tbl, on="key").drop(columns=["key"])

    pairs["distance_m"] = pairs["rep_point"].distance(pairs["geometry"])

    # --- 4) Vælg top-K nærmeste stationer pr region ---
    pairs = pairs.sort_values(["region_code", "distance_m"])
    topk = pairs.groupby("region_code", as_index=False).head(K).copy()

    # --- 5) IDW weights ---
    eps = 1e-9
    topk["distance_m_safe"] = topk["distance_m"].clip(lower=eps)
    topk["idw"] = 1.0 / np.power(topk["distance_m_safe"], p)

    def normalize_group(g: pd.DataFrame) -> pd.DataFrame:
        if (g["distance_m"] <= 1e-6).any():
            z = g["distance_m"] <= 1e-6
            g = g.copy()
            g["weight"] = 0.0
            g.loc[z, "weight"] = 1.0 / z.sum()
            return g
        s = g["idw"].sum()
        g = g.copy()
        g["weight"] = g["idw"] / s if s > 0 else 1.0 / len(g)
        return g

    region_station_weights = (
        topk.groupby("region_code", group_keys=False)
            .apply(normalize_group)
            .loc[:, ["region_code", "station_id", "distance_m", "weight"]]
            .reset_index(drop=True)
    )

    # sanity checks
    chk = region_station_weights.groupby("region_code")["weight"].sum()
    if not np.allclose(chk.values, 1.0, atol=1e-8):
        bad = chk[~np.isclose(chk.values, 1.0, atol=1e-8)]
        raise RuntimeError(f"Vægte summer ikke til 1 for disse regioner:\n{bad}")

    counts = region_station_weights.groupby("region_code")["station_id"].count()
    if (counts < min(K, region_station_weights["station_id"].nunique())).any():
        print("ADVARSEL: Nogle regioner har færre end K stationer (typisk fordi station-universet er lille).")

    return region_station_weights

In [5]:
gdf_regions = gpd.read_file("Meta/regions-1000m.geojson")

region_station_weights = build_region_station_weights_idw(
    df_stations=df_stations,  # already country='FR'
    gdf_regions=gdf_regions,
    reg_code_col="code",
    K=5,
    p=2.0
)

print(sorted(region_station_weights["region_code"].unique()))
# -> should be only the 13 mainland region codes
region_station_weights.to_csv("region_station_KNN_weights_mainland.csv", index=False)

['11', '24', '27', '28', '32', '44', '52', '53', '75', '76', '84', '93', '94']


/var/folders/g6/v5w89nm96b732r3v5w91v8dm0000gn/T/ipykernel_75106/527882002.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(normalize_group)
